# US Stocks Volatility Analysis - Version 2: GARCH Modeling\n\n**Author:** Trade Vectors LLP\n**Date:** March 2026\n\n## Overview\nThis notebook extends the basic volatility analysis by implementing GARCH (Generalized Autoregressive Conditional Heteroskedasticity) models for volatility forecasting:\n- GARCH(1,1) Model\n- EGARCH Model (Exponential GARCH)\n- GJR-GARCH Model (Asymmetric GARCH)\n- Volatility Forecasting and Prediction\n\n## Data Source\nHistorical stock price data (NVDA, AAPL, MSFT, AMZN, IBM) from the USStocks folder.

In [ ]:
# Install required GARCH modeling library\n# !pip install arch\n\n# Import libraries\nimport pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport matplotlib.dates as mdates\nimport seaborn as sns\nfrom arch import arch_model\nfrom scipy import stats\nimport warnings\nwarnings.filterwarnings('ignore')\n\nplt.style.use('seaborn-v0_8-darkgrid')\nprint('Libraries loaded successfully')

In [ ]:
# Load and prepare stock data\ndef load_stock_data(ticker):\n    """Load historical stock data from CSV file"""\n    df = pd.read_csv(f'../USStocks/{ticker}_STK.csv')\n    df['Date'] = pd.to_datetime(df['Date'])\n    df.set_index('Date', inplace=True)\n    df.sort_index(inplace=True)\n    # Calculate log returns scaled by 100 (percent returns for GARCH)\n    df['log_return'] = np.log(df['Close'] / df['Close'].shift(1)) * 100\n    df.dropna(inplace=True)\n    return df\n\ntickers = ['NVDA', 'AAPL', 'MSFT', 'AMZN', 'IBM']\nstock_data = {}\nfor ticker in tickers:\n    try:\n        stock_data[ticker] = load_stock_data(ticker)\n        print(f'{ticker}: {len(stock_data[ticker])} rows, Returns range: {stock_data[ticker]["log_return"].min():.2f}% to {stock_data[ticker]["log_return"].max():.2f}%')\n    except Exception as e:\n        print(f'Error loading {ticker}: {e}')

In [ ]:
# Fit GARCH(1,1) Model\ndef fit_garch_model(returns, model_type='GARCH', p=1, q=1):\n    """\n    Fit GARCH family models\n    model_type: 'GARCH', 'EGARCH', 'GJR-GARCH'\n    """\n    if model_type == 'GARCH':\n        model = arch_model(returns, vol='Garch', p=p, q=q, dist='normal')\n    elif model_type == 'EGARCH':\n        model = arch_model(returns, vol='EGARCH', p=p, q=q, dist='normal')\n    elif model_type == 'GJR-GARCH':\n        model = arch_model(returns, vol='Garch', p=p, o=1, q=q, dist='normal')\n    \n    result = model.fit(disp='off')\n    return result\n\n# Fit GARCH(1,1) for each stock\ngarch_results = {}\nfor ticker in stock_data.keys():\n    returns = stock_data[ticker]['log_return']\n    result = fit_garch_model(returns, model_type='GARCH')\n    garch_results[ticker] = result\n    \n    print(f'{ticker} GARCH(1,1) Results:')\n    print(f'  AIC: {result.aic:.4f}')\n    print(f'  BIC: {result.bic:.4f}')\n    print(f'  Persistence (alpha+beta): {result.params["alpha[1]"] + result.params["beta[1]"]:.4f}')\n    print()

In [ ]:
# Compare GARCH model types for NVDA\nticker = 'NVDA'\nreturns = stock_data[ticker]['log_return']\n\nmodel_comparison = {}\nfor model_type in ['GARCH', 'EGARCH', 'GJR-GARCH']:\n    result = fit_garch_model(returns, model_type=model_type)\n    model_comparison[model_type] = {'AIC': result.aic, 'BIC': result.bic, 'result': result}\n    print(f'{model_type}: AIC={result.aic:.2f}, BIC={result.bic:.2f}')\n\nbest_model = min(model_comparison, key=lambda x: model_comparison[x]['AIC'])\nprint(f'\nBest model by AIC for {ticker}: {best_model}')

In [ ]:
# Volatility Forecasting - Next 30 days\nforecast_horizon = 30\nvolatility_forecasts = {}\n\nfor ticker in garch_results.keys():\n    result = garch_results[ticker]\n    # Forecast next 30 days\n    forecast = result.forecast(horizon=forecast_horizon)\n    \n    # Convert variance forecast to annualized volatility\n    forecast_vol = np.sqrt(forecast.variance.iloc[-1]) * np.sqrt(252) / 100\n    volatility_forecasts[ticker] = forecast_vol\n    \n    print(f'{ticker}:')\n    print(f'  1-day forecast vol: {forecast_vol.iloc[0]:.2%}')\n    print(f'  30-day forecast vol: {forecast_vol.iloc[-1]:.2%}')\n    print()

In [ ]:
# Plot conditional volatility for all stocks\nfig, axes = plt.subplots(3, 2, figsize=(16, 12))\naxes = axes.flatten()\n\nfor i, ticker in enumerate(garch_results.keys()):\n    result = garch_results[ticker]\n    cond_vol = result.conditional_volatility * np.sqrt(252) / 100\n    \n    axes[i].plot(stock_data[ticker].index[-len(cond_vol):], cond_vol, color='blue', linewidth=1.5)\n    axes[i].set_title(f'{ticker} - GARCH(1,1) Conditional Volatility', fontsize=12, fontweight='bold')\n    axes[i].set_xlabel('Date')\n    axes[i].set_ylabel('Annualized Volatility')\n    axes[i].grid(True, alpha=0.3)\n\naxes[-1].set_visible(False)  # Hide last empty subplot\nplt.suptitle('GARCH(1,1) Conditional Volatility - All Stocks', fontsize=16, fontweight='bold', y=1.02)\nplt.tight_layout()\nplt.show()

In [ ]:
# Volatility Clustering Analysis\nplt.figure(figsize=(12, 6))\n\nticker = 'NVDA'\ndf = stock_data[ticker]\nreturns = df['log_return']\n\nplt.subplot(2, 1, 1)\nplt.plot(df.index, returns, color='navy', linewidth=0.8, alpha=0.7)\nplt.title(f'{ticker} Log Returns - Volatility Clustering', fontsize=14)\nplt.ylabel('Log Returns (%)')\nplt.grid(True, alpha=0.3)\n\nplt.subplot(2, 1, 2)\ncond_vol = garch_results[ticker].conditional_volatility\nplt.plot(df.index[-len(cond_vol):], cond_vol, color='red', linewidth=1.5)\nplt.title('Conditional Volatility from GARCH(1,1)', fontsize=14)\nplt.ylabel('Conditional Volatility')\nplt.grid(True, alpha=0.3)\n\nplt.tight_layout()\nplt.show()

## Summary\n\nGARCH models capture key features of financial volatility:\n1. **Volatility clustering**: Large changes tend to follow large changes\n2. **Mean reversion**: Volatility returns to long-run average\n3. **Persistence**: High alpha+beta indicates slow mean reversion\n\n## Model Selection Criteria\n- GARCH(1,1): Standard baseline, good for most stocks\n- EGARCH: Captures leverage effect (bad news increases volatility more)\n- GJR-GARCH: Asymmetric response to positive/negative shocks\n\n## Next Steps\n- Extend to multivariate GARCH (DCC-GARCH)\n- Implement rolling window GARCH for dynamic volatility\n- Use GARCH forecasts for options pricing\n- VaR estimation using GARCH conditional volatility